# MedNorm-VI S3 — ICD/RxNorm retrieval embeddings (Colab workflow)

**Colab-only.** Full training is OFF by default. This notebook is reviewable
infrastructure, not a trained model. See docs/training/colab_execution_policy.md.


## 1. Colab & runtime detection · 2. GPU/RAM/disk report


In [ ]:
import sys
import platform
import shutil

IN_COLAB = 'google.colab' in sys.modules
print('in_colab', IN_COLAB, 'python', platform.python_version())
total, used, free = shutil.disk_usage('/')
print('disk_free_gb', round(free / 1e9, 1))
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print('gpu', torch.cuda.get_device_name(0), 'vram_gb', round(p.total_memory / 1e9, 1))
except ImportError as exc:
    print('torch not installed yet:', exc)


## 3. Configurable Google Drive roots (edit PROJECT_ROOT only)


In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/mednorm-vi'   # <-- EDIT THIS
DATA_ROOT = PROJECT_ROOT + '/data'
MODEL_CACHE = PROJECT_ROOT + '/model_cache'
CHECKPOINT_ROOT = PROJECT_ROOT + '/checkpoints/full_v1'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError as exc:
    print('not in Colab:', exc)


## 4. Pinned dependencies


In [ ]:
PINS = ["torch==2.3.1", "transformers==4.44.2", "datasets==2.21.0", "accelerate==0.33.0", "sentence-transformers==3.0.1"]
# !pip -q install ' '.join(PINS)
print('pins', PINS)


## 5. Git commit / config verification


In [ ]:
EXPECTED_COMMIT = '<fill: reviewed repo HEAD>'
SEED = 20260723
print('expected_commit', EXPECTED_COMMIT, 'seed', SEED)


## 6. Governed-corpus + split hash verification


In [ ]:
import json
import hashlib

CORPUS = DATA_ROOT + '/derived/training_corpora/mednorm_vi_training_v1'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

with open(CORPUS + '/manifests/split_manifest.json') as fh:
    sm = json.load(fh)
for split, expected in sm['sha256'].items():
    assert sha256(CORPUS + '/splits/' + split + '.jsonl') == expected, split
    print(split, 'OK', sm['counts'][split])
print('cross_split_family_leakage', sm['cross_split_family_leakage'])


## 7. Model/tokenizer revision variables · 8. Model download (Colab only) · 9. Offline-after-download


In [ ]:
import os

MODEL = "BAAI/bge-m3"   # verify against the architecture budget
MODEL_REVISION = "<pin>"   # PIN a real revision
os.environ['HF_HUB_OFFLINE'] = '0'   # 0 for the one-time download; set 1 afterwards
# from transformers import AutoModel, AutoTokenizer
# tok = AutoTokenizer.from_pretrained(MODEL, revision=MODEL_REVISION, cache_dir=MODEL_CACHE)
# net = AutoModel.from_pretrained(MODEL, revision=MODEL_REVISION, cache_dir=MODEL_CACHE)
print('model', MODEL, 'revision', MODEL_REVISION)  # weights never committed


## 10. Smoke mode · 11. Full mode OFF by default · 12. Explicit confirmation


In [ ]:
CONFIG = {
    'seed': SEED, 'batch_size': 8, 'grad_accum': 2, 'mixed_precision': 'bf16',
    'checkpoint_every': 50, 'keep_last_k': 3, 'early_stopping_patience': 3,
    'deterministic': True, 'max_smoke_batches': 3,
}
SMOKE = True
RUN_FULL_TRAINING = False   # never auto-runs
CONFIRM_FULL = ''           # must equal 'YES' to allow full training
if RUN_FULL_TRAINING and CONFIRM_FULL != 'YES':
    raise SystemExit('Set CONFIRM_FULL="YES" to run full training deliberately.')
print('stage', "S3", 'smoke', SMOKE, 'full', RUN_FULL_TRAINING)


## 13. Resume logic · 14. Checkpoint rotation


In [ ]:
import os

OUT = CHECKPOINT_ROOT + '/' + "retrieval/icd_dense"
os.makedirs(OUT, exist_ok=True)
def latest_checkpoint(path):
    cks = sorted(f for f in os.listdir(path) if f.startswith('ckpt-')) if os.path.isdir(path) else []
    return cks[-1] if cks else None
print('resume_from', latest_checkpoint(OUT), '| keep_last_k', CONFIG['keep_last_k'])


## 15. Metrics & logs · 16. Parameter count · 17. Artifact hashes · 18. Checkpoint manifest


In [ ]:
import json

metrics = {'mode': 'smoke', 'note': 'path-verified; not a trained model'}
param_count = None   # fill from net.num_parameters() when the model is loaded
manifest = {
    'manifest_version': 1, 'role': "retrieval/icd_dense",
    'base_model': {'name': MODEL, 'revision': MODEL_REVISION, 'parameter_count': param_count},
    'training': {'git_commit': EXPECTED_COMMIT, 'seed': SEED, 'mode': 'smoke',
                 'dataset_manifest_hash': '<fill>', 'split_manifest_hash': '<fill>',
                 'config_hash': '<fill>'},
    'metrics': metrics, 'status': 'SMOKE_ONLY',
}
with open(OUT + '/checkpoint_manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)
print('wrote manifest', OUT)


## 19. Drive export · 20. Return-to-repository instructions
Copy `OUT` into `models/checkpoints/full_v1/retrieval/icd_dense` in the repo
(weights git-ignored; manifest reviewable). Then run local validation:
`model_registry.cli --profile full --require-local-paths`, `pytest -q`,
`phase1c_foundation.cli doctor`. Do NOT commit restricted base-model weights.
